# Decorators, Metaprogramming & Descriptors (5+ Years Interview Guide)
Exhaustive revision guide with dedicated separate cells for @functools.wraps, parameterized decorator factories, __call__ class decorators, descriptors (__get__, __set__, __delete__), __new__, and __init_subclass__ on transaction classes.

### Key 5-Year Interview Concepts Covered:
- **Function Wrapping**: Dedicated separate cell for `@functools.wraps` preserving function signatures and docstrings.
- **Decorator Factories**: Dedicated separate cell for parameterized 3-level decorator closures.
- **Class Decorators**: Dedicated separate cell for class-based decorators implementing `__call__()`.
- **Descriptors**: Dedicated separate cells for `__get__`, `__set__`, and `__delete__` access control.
- **Metaprogramming**: Dedicated separate cells for `__new__()` and `__init_subclass__()`.

This interactive revision guide uses `data/raw_transactions.csv` with 1-to-1 atomic method cells.

In [1]:
# Setup imports & dataset loading from raw_transactions.csv
import csv
import sys
import time
import os
import functools
import contextlib
import asyncio
import threading
from dataclasses import dataclass
from typing import List, Dict, Optional, Union, Protocol, Literal, Final, TypedDict, Callable, TypeVar

csv_path = 'data/raw_transactions.csv' if os.path.exists('data/raw_transactions.csv') else '../data/raw_transactions.csv'
transactions = []
with open(csv_path, mode='r', encoding='utf-8') as f:
    reader = csv.DictReader(f)
    for row in reader:
        transactions.append(row)

print(f"Python Version: {sys.version.split()[0]}")
print(f"Loaded {len(transactions)} transaction records from {csv_path}")

Python Version: 3.12.7
Loaded 15000 transaction records from data/raw_transactions.csv


### Function Decorators: `@functools.wraps`
**Explanation**: Wraps functions while preserving identity (`__name__`, `__doc__`, annotations).

**Syntax**: `@functools.wraps(func)`

In [2]:
def log_execution(func):
    @functools.wraps(func)
    def wrapper(*args, **kwargs):
        t0 = time.perf_counter()
        res = func(*args, **kwargs)
        print(f'[AUDIT] {func.__name__} completed in {(time.perf_counter()-t0)*1000:.3f} ms')
        return res
    return wrapper

@log_execution
def process_tx(tx_id):
    """Settles transaction record."""
    return f'Settled {tx_id}'

print(process_tx(transactions[0]['transaction_id']))
print('Preserved docstring:', process_tx.__doc__)

[AUDIT] process_tx completed in 0.002 ms
Settled TX110686
Preserved docstring: Settles transaction record.


### Parameterized Decorator Factories
**Explanation**: 3-level closure taking configuration parameters for the decorator.

**Syntax**: `@require_limit(threshold=500.0)`

In [3]:
def check_limit(max_limit):
    def decorator(func):
        @functools.wraps(func)
        def wrapper(amount, *args, **kwargs):
            if amount > max_limit:
                return f'REJECTED: ${amount} exceeds limit ${max_limit}'
            return func(amount, *args, **kwargs)
        return wrapper
    return decorator

@check_limit(1000.0)
def authorize_payment(amt):
    return f'AUTHORIZED ${amt}'

print(authorize_payment(500.0))
print(authorize_payment(1500.0))

AUTHORIZED $500.0
REJECTED: $1500.0 exceeds limit $1000.0


### Class-Based Decorators: `__call__()`
**Explanation**: Decorator implemented as a class using `__call__` for stateful tracking.

**Syntax**: `class StatefulDecorator: def __call__(self, *args): ...`

In [4]:
class CallCounter:
    def __init__(self, func):
        self.func = func
        self.count = 0
        functools.update_wrapper(self, func)
    def __call__(self, *args, **kwargs):
        self.count += 1
        return self.func(*args, **kwargs)

@CallCounter
def handle_webhook(): return 'OK'

handle_webhook(); handle_webhook()
print('Total calls tracked by class decorator:', handle_webhook.count)

Total calls tracked by class decorator: 2


### Descriptor Protocol: `__get__`
**Explanation**: Intercepts attribute retrieval on instance/owner class.

**Syntax**: `def __get__(self, instance, owner): ...`

In [5]:
class ReadOnlyDescriptor:
    def __init__(self, val): self.val = val
    def __get__(self, instance, owner): return self.val

class Config:
    fee_rate = ReadOnlyDescriptor(0.025)

print('Descriptor __get__ fee_rate:', Config().fee_rate)

Descriptor __get__ fee_rate: 0.025


### Descriptor Protocol: `__set__` & `__delete__`
**Explanation**: Intercepts attribute assignment and deletion for data validation.

**Syntax**: `def __set__(self, instance, value): ...` / `def __delete__(self, instance): ...`

In [6]:
class NonNegative:
    def __set_name__(self, owner, name): self.name = name
    def __get__(self, instance, owner): return instance.__dict__.get(self.name, 0.0)
    def __set__(self, instance, value):
        if value < 0: raise ValueError('Cannot be negative!')
        instance.__dict__[self.name] = value

class Account:
    balance = NonNegative()

acc = Account()
acc.balance = 500.0
print('Validated balance:', acc.balance)

Validated balance: 500.0


### Object Instantiation Hook: `__new__()`
**Explanation**: Static method responsible for allocating raw object memory before `__init__` executes.

**Syntax**: `def __new__(cls, *args, **kwargs): return super().__new__(cls)`

In [7]:
class SingletonService:
    _instance = None
    def __new__(cls):
        if cls._instance is None:
            cls._instance = super().__new__(cls)
        return cls._instance

s1 = SingletonService()
s2 = SingletonService()
print('Are singleton instances identical (s1 is s2)?:', s1 is s2)

Are singleton instances identical (s1 is s2)?: True


### Subclass Interception Hook: `__init_subclass__()`
**Explanation**: PEP 487 hook customizing and registering subclasses at definition time without metaclasses.

**Syntax**: `def __init_subclass__(cls, **kwargs): ...`

In [8]:
class PaymentGateway:
    gateways = {}
    def __init_subclass__(cls, gateway_name=None, **kwargs):
        super().__init_subclass__(**kwargs)
        if gateway_name: cls.gateways[gateway_name] = cls

class StripeGateway(PaymentGateway, gateway_name='Stripe'): pass
class PayPalGateway(PaymentGateway, gateway_name='PayPal'): pass
print('Auto-Registered Gateways via __init_subclass__:', PaymentGateway.gateways)

Auto-Registered Gateways via __init_subclass__: {'Stripe': <class '__main__.StripeGateway'>, 'PayPal': <class '__main__.PayPalGateway'>}


## Section: Senior Fintech Interview Scenarios (5+ Years Experience)

### Q1: Python Function Binding Protocol via Descriptors
**Explanation**: Explain that functions implement `__get__`. Accessing `inst.method` invokes `func.__get__(inst, Class)`, binding `self`.

**Syntax**: `bound_method = obj.func; bound_method.__self__ is obj`

In [9]:
print('Functions implement descriptor __get__ to bind instance `self` at runtime.')

Functions implement descriptor __get__ to bind instance `self` at runtime.
